# Main grid — granite-4.2-8b
Research plan §11/§12 (this-week scope), DECISIONS D24/D28/D29/D30. AgentDojo 0.1.35 (suites v1.2.2),
`MultilingualAttack` conditions **en, bn_mt, bn_llm, banglish_llm** over the **150-pair** nested subset,
no defence, plus the 97 no-attack utility episodes. Payloads frozen at 150/150 per condition (D28).

**Model:** `ibm-granite/granite-4.2-8b-GGUF` → `granite-4.2-8b-Q4_K_M.gguf` @ `93f3f6a8938e` (5.35 GB, SHA-256 `16a9369d0805f80b…` asserted). **Serve:** Q4_K_M; 2 replicas (one per T4), thinking OFF (D21 primary variant, 1 stream/replica). Variant `no_think_kwargs`, `-c 24576 -np 1 --jinja --temp 0 --seed 20261002`, llama.cpp `b11344`, 1 agent stream per server (as in the pilot, D21).

**Estimate:** 697 episodes (150 pairs × 4 conditions + 97 no-attack utility) at the pilot rate 252.9 eps/h × 1.1 + 0.25 h setup ≈ **3.28 h** (at half the pilot rate: 6.31 h). Pilot rates were measured on no-attack banking+slack; attack episodes and workspace/travel may be slower. A deadline guard stops new episodes at 11 h so the commit always finishes and saves.

**Self-contained:** cell 1 unpacks the embedded harness (tarball 94,925 B gz, 163 files,
sha256 `ee7b4195e694977b3f4e060155d66ca373027cdc2dee98ee88750b5305169334`) into `/kaggle/working/code` and verifies `PAYLOAD_SHA256SUMS.txt`; it raises on any mismatch.
No dataset or git clone needed.

**Outputs (/kaggle/working):** `runs/<run_id>.jsonl` (one scored record per episode, skip-if-done keyed),
`runs/traces/<label>/…` (AgentDojo-format trace per episode), `<run_id>_traces.zip`,
`runs_manifest_<run_id>.json` (provenance: embed sha256, payload sums, GGUF sha256, serve command, pins),
`server_*.log`, and a final `MAIN_RESULT …` line.

**Resume:** if a run stops early, attach its Output as an input (Add Input → Notebook Output) and
Save & Run All again; finished episodes are skipped.

Settings: Accelerator **GPU T4 x2**, Internet **on**, notebook **private**, then **Save Version → Save & Run All (Commit)**.
Generated by `scripts/build_main_grid_notebook.py`; prepared 2026-10-06, **not run**.

In [ ]:
# ===== 1. Unpack the embedded harness + verify PAYLOAD_SHA256SUMS (fails loudly) =====
# Self-contained: no Kaggle dataset or git clone. Built by scripts/build_main_grid_notebook.py.
import base64, hashlib, io, json, os, shutil, sys, tarfile, time
from pathlib import Path
SESSION_T0 = time.time()
CODE_DIR = '/kaggle/working/code'
EMBED_SHA256 = 'ee7b4195e694977b3f4e060155d66ca373027cdc2dee98ee88750b5305169334'
EMBED_BYTES = 94925
EMBED_N_FILES = 163
EMBED_B64 = '''
...
'''

_blob = base64.b64decode(''.join(EMBED_B64.split()))
EMBED_SHA256_GOT = hashlib.sha256(_blob).hexdigest()
print(f'EMBED tarball: {len(_blob)} B, sha256 {EMBED_SHA256_GOT}', flush=True)
if len(_blob) != EMBED_BYTES or EMBED_SHA256_GOT != EMBED_SHA256:
    raise RuntimeError(f'EMBEDDED TARBALL CORRUPT: got {len(_blob)} B sha256 {EMBED_SHA256_GOT}, '
                       f'expected {EMBED_BYTES} B sha256 {EMBED_SHA256}. Re-import the notebook.')
if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)          # always run exactly the embedded tree
os.makedirs(CODE_DIR)
with tarfile.open(fileobj=io.BytesIO(_blob), mode='r:gz') as _tf:
    _members = _tf.getmembers()
    for _m in _members:
        if not _m.isfile() or _m.name.startswith('/') or '..' in _m.name.split('/'):
            raise RuntimeError(f'unsafe tar member {_m.name!r}')
    if hasattr(tarfile, 'data_filter'):
        _tf.extractall(CODE_DIR, members=_members, filter='data')
    else:
        _tf.extractall(CODE_DIR, members=_members)
_root = Path(CODE_DIR)
_sha = lambda b: hashlib.sha256(b).hexdigest()
_bad = []

# (a) every embedded file vs EMBED_MANIFEST.json, and nothing extra
EMBED_MANIFEST = json.loads((_root / 'EMBED_MANIFEST.json').read_text(encoding='utf-8'))
for _rel, _meta in sorted(EMBED_MANIFEST['files'].items()):
    _p = _root / _rel
    _got = _sha(_p.read_bytes()) if _p.is_file() else 'MISSING'
    if _got != _meta['sha256']:
        _bad.append(f'manifest: {_rel} sha256 {_got[:16]} != {_meta["sha256"][:16]}')
_extra = sorted(str(p.relative_to(_root)) for p in _root.rglob('*') if p.is_file()
                and str(p.relative_to(_root)) not in EMBED_MANIFEST['files'] and p.name != 'EMBED_MANIFEST.json')
if _extra:
    _bad.append(f'manifest: unexpected files {_extra[:5]}')

# (b) PAYLOAD_SHA256SUMS.txt: every frozen payload file + CANONICAL goal-map hash of payloads.json
_C = _root / 'data' / 'conditions'
SUMS_TEXT = (_C / 'PAYLOAD_SHA256SUMS.txt').read_text(encoding='utf-8')
PAYLOAD_SUMS_SHA256 = _sha(SUMS_TEXT.encode('utf-8'))
_pay = json.loads((_root / 'data' / 'payloads.json').read_text(encoding='utf-8'))
_n_files = _n_canon = 0
_sum_conds = set()
for _line in SUMS_TEXT.splitlines():
    if not _line.strip() or _line.startswith('#'):
        continue
    _digest, _name = _line.split('  ', 1)
    if _name.startswith('CANONICAL_GOALS_JSON('):
        _cond = _name[len('CANONICAL_GOALS_JSON('):-1]
        _got = _sha(json.dumps(_pay['conditions'][_cond]['goals'], ensure_ascii=False, sort_keys=True,
                               separators=(',', ':')).encode('utf-8'))
        _n_canon += 1
        _sum_conds.add(_cond)
    else:
        _p = _C / _name
        _got = _sha(_p.read_bytes()) if _p.is_file() else 'MISSING'
        _n_files += 1
    if _got != _digest:
        _bad.append(f'PAYLOAD_SHA256SUMS: {_name} sha256 {_got[:16]} != {_digest[:16]}')

# (c) the texts the attack loads at run time (payloads.json) == the sha-verified files
def _body(_t):
    _ls = _t.split('\n'); _i = 0
    while _i < len(_ls) and _ls[_i].startswith('#'):
        _i += 1
    return '\n'.join(_ls[_i:]).strip()
_n_goal_files = 0
for _cond in sorted(_sum_conds):
    for _p in sorted((_C / _cond).glob('*__injection_task_*.txt')):
        _key = _p.stem.replace('__', '/')
        if _body(_p.read_text(encoding='utf-8')) != str(_pay['conditions'][_cond]['goals'].get(_key, '')).strip():
            _bad.append(f'payloads.json {_cond} {_key} != sha-verified file')
        _n_goal_files += 1
    _w = (_C / '_wrappers' / f'W1__{_cond}.txt').read_text(encoding='utf-8')
    if _body(_w) != str(_pay['wrappers']['W1'].get(_cond, '')).strip():
        _bad.append(f'payloads.json W1 wrapper {_cond} != sha-verified file')

if _bad:
    print('\n'.join(_bad), flush=True)
    raise RuntimeError(f'PAYLOAD/EMBED VERIFICATION FAILED ({len(_bad)} problems) - do not run the grid')
PAYLOAD_VERIFY = {'files': _n_files, 'canonical': _n_canon, 'goal_files_vs_payloads_json': _n_goal_files,
                  'wrappers': len(_sum_conds), 'conditions': sorted(_sum_conds), 'sums_sha256': PAYLOAD_SUMS_SHA256}
print(f'EMBED OK: {len(EMBED_MANIFEST["files"])} files unpacked to {CODE_DIR} (manifest verified)', flush=True)
print(f'PAYLOAD_SHA256SUMS OK: {_n_files} files + {_n_canon} canonical goal maps; '
      f'{_n_goal_files} goal texts + {len(_sum_conds)} W1 wrappers in payloads.json match; '
      f'sums file sha256 {PAYLOAD_SUMS_SHA256}', flush=True)
sys.path.insert(0, CODE_DIR)
os.environ.pop('BANGLA_PI_PAYLOADS', None)   # attacks must load the verified data/payloads.json


In [ ]:
# ===== 2. PARAMETERS =====
MODEL_KEY  = 'granite'          # gemma | gptoss | ministral | granite | qwen
SUBSET     = '150'             # '150' (nested, §12 this week) or '300' (§11)
CONDITIONS = ['en', 'bn_mt', 'bn_llm', 'banglish_llm']
DEFENSE    = None              # None, or e.g. 'tool_filter' (single defence, one model)
RUN_UTILITY = True             # + the 97 no-attack utility episodes (condition 'none')
SHARD, N_SHARDS = 1, 1          # disjoint pair/utility shards (union = full grid)
STREAMS_PER_SERVER = 1         # = the pilots: 1 agent stream per llama-server (D17/D20/D21)
DEADLINE_H = 11.0              # no NEW episodes after this many hours since cell 1 (Kaggle kills at 12 h)
SMOKE_N    = None              # smoke test only: first N attack + first N utility episodes; None = full grid
WORK       = '/kaggle/working'
OUTDIR     = '/kaggle/working/runs'
INPUT_ROOT = '/kaggle/input'   # attach a previous version's Output here -> skip-if-done resume
EST_SESSION_H = 3.28        # builder estimate (pilot rate x1.10 + setup); see header

from runners.main_grid import (MODELS, resolve_model, LLAMA_BUILD, SEED, BENCHMARK_VERSION,
                               episode_count)
MODEL_KEY = resolve_model(MODEL_KEY)
M = MODELS[MODEL_KEY]
LABEL = M['label'] + (f'-{DEFENSE}' if DEFENSE else '')
RUN_ID = (f'{MODEL_KEY}_{SUBSET}' + (f'_{DEFENSE}' if DEFENSE else '')
          + (f'_shard{SHARD}of{N_SHARDS}' if N_SHARDS > 1 else ''))
PAIRS_PATH = f'{CODE_DIR}/subset/pairs_150.json' if SUBSET == '150' else f'{CODE_DIR}/subset/pairs.json'
PAIRS = json.loads(Path(PAIRS_PATH).read_text(encoding='utf-8'))['pairs']
# every pair must have a filled payload (and W1 wrapper) in every condition (D28: 150/150 x 4)
_pay = json.loads(Path(f'{CODE_DIR}/data/payloads.json').read_text(encoding='utf-8'))
_filled = lambda v: isinstance(v, str) and v.strip() != '' and not v.strip().startswith('TODO_')
_miss = [(c, p['suite'], p['injection_task']) for c in CONDITIONS for p in PAIRS
         if not _filled(_pay['conditions'][c]['goals'].get(f"{p['suite']}/{p['injection_task']}"))
         or not _filled(_pay['wrappers']['W1'].get(c))]
if _miss:
    raise RuntimeError(f'{len(_miss)} (condition, pair) payloads missing, e.g. {_miss[:3]}')
assert set(CONDITIONS) <= set(PAYLOAD_VERIFY['conditions']), 'condition not covered by PAYLOAD_SHA256SUMS'
print(f'{RUN_ID}: {LABEL} layout={M["layout"]} variant={M["variant"]} subset={SUBSET} '
      f'({len(PAIRS)} pairs, coverage {len(PAIRS)}/{len(PAIRS)} x {len(CONDITIONS)}) conditions={CONDITIONS} '
      f'defense={DEFENSE} utility={RUN_UTILITY} shard={SHARD}/{N_SHARDS}', flush=True)
print(json.dumps(M, indent=1), flush=True)


In [ ]:
# ===== 3. GPU check (needs Accelerator = GPU T4 x2) =====
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)
_n = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.count('GPU ')
print(f'GPUs visible: {_n}', flush=True)
assert _n >= 2, 'Need Accelerator = GPU T4 x2'


In [ ]:
# ===== 4. Pinned llama.cpp (CUDA 12.8 build) + agentdojo 0.1.35 (same as the pilots) =====
import glob
LLAMA_DIR = '/kaggle/tmp/llama' if os.path.isdir('/kaggle/tmp') else '/tmp/llama'   # keep Output small
os.makedirs(LLAMA_DIR, exist_ok=True)
_rel = f'https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}'
for _url in (f'{_rel}/cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz',
             f'{_rel}/llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz'):
    _fn = f"{LLAMA_DIR}/{_url.split('/')[-1]}"
    subprocess.run(['wget', '-q', '-O', _fn, _url], check=True)
    subprocess.run(['tar', 'xzf', _fn, '-C', LLAMA_DIR], check=True)
SERVER = glob.glob(f'{LLAMA_DIR}/**/llama-server', recursive=True)[0]
print('llama-server at', SERVER, flush=True)
print(subprocess.run([SERVER, '--version'], capture_output=True, text=True).stderr[:300], flush=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'agentdojo==0.1.35', 'openai==3.23.0'], check=True)


In [ ]:
# ===== 5. Pinned GGUF -> scratch (never /kaggle/working, D13); assert size + SHA-256 =====
from huggingface_hub import hf_hub_download
for _p in glob.glob('/kaggle/working/models/**/*', recursive=True):
    if _p.endswith(('.gguf', '.incomplete', '.lock')) and os.path.isfile(_p):
        os.remove(_p)
MODELS_DIR = '/kaggle/tmp/models' if os.path.isdir('/kaggle/tmp') else '/tmp/models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['HF_HUB_CACHE'] = os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR
_free = shutil.disk_usage(MODELS_DIR).free / 2**30
print(f'MODELS_DIR={MODELS_DIR} free={_free:.1f} GiB', flush=True)
assert _free > M['min_free_gib'], f"scratch too small for {M['file']}"
_t0 = time.time()
MODEL_PATH = hf_hub_download(M['repo'], M['file'], revision=M['rev'], local_dir=MODELS_DIR)
assert os.path.getsize(MODEL_PATH) == M['size'], f'size mismatch {os.path.getsize(MODEL_PATH)} != {M["size"]}'
_h = hashlib.sha256()
with open(MODEL_PATH, 'rb') as _f:
    while (_b := _f.read(1 << 22)):
        _h.update(_b)
GGUF_SHA = _h.hexdigest()
print(f"{M['file']} {M['size']} B sha256 {GGUF_SHA} match={GGUF_SHA == M['sha256']} ({time.time()-_t0:.0f}s)", flush=True)
assert GGUF_SHA == M['sha256'], 'GGUF sha256 mismatch'


In [ ]:
# ===== 6. Serve in the layout + variant that passed the pilot; prove template kwargs =====
import requests
from runners.main_grid import server_commands, template_ok
PROCS = []
def _wait(p, port, timeout_s=900):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if p.poll() is not None:
            return False
        try:
            if requests.get(f'http://127.0.0.1:{port}/v1/models', timeout=5).ok:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False
def _template_tail(port):
    try:
        r = requests.post(f'http://127.0.0.1:{port}/apply-template', timeout=30,
                          json={'messages': [{'role': 'user', 'content': 'ping'}]})
        return r.json().get('prompt', '')
    except Exception as e:
        return f'<apply-template failed: {e}>'
SPECS = server_commands(M, SERVER, MODEL_PATH, SEED, log_dir=WORK)
for _s in SPECS:
    print(f"serve[{M['variant']}] gpu={_s['gpu']} port={_s['port']}: " + ' '.join(_s['cmd'][1:]), flush=True)
    _t0 = time.time()
    _p = subprocess.Popen(_s['cmd'], env=dict(os.environ, **_s['env']), stdout=open(_s['log'], 'w'),
                          stderr=subprocess.STDOUT)
    PROCS.append(_p)
    if not _wait(_p, _s['port']):
        for _q in PROCS:
            _q.kill()
        raise RuntimeError(f"llama-server on :{_s['port']} failed (rc={_p.poll()}):\n" + open(_s['log']).read()[-2000:])
    print(f"  up on :{_s['port']} ({time.time()-_t0:.0f}s)", flush=True)
PORTS = [_s['port'] for _s in SPECS]
TEMPLATE_TAILS = {p: _template_tail(p) for p in PORTS}
for _p, _t in TEMPLATE_TAILS.items():
    print(f'template[{_p}] tail: {_t[-200:]!r}', flush=True)
TEMPLATE_OK = all(template_ok(M, t) for t in TEMPLATE_TAILS.values())
print(f"TEMPLATE_OK={TEMPLATE_OK} expect={M['template_expect']!r} forbid={M['template_forbid']!r}", flush=True)
if not TEMPLATE_OK:
    for _q in PROCS:
        _q.kill()
    raise RuntimeError('chat-template kwargs did not take effect (thinking/reasoning setting) - not running the grid')
MODEL_ID = requests.get(f'http://127.0.0.1:{PORTS[0]}/v1/models', timeout=10).json()['data'][0]['id']
SERVE_USED = {'layout': M['layout'], 'variant': M['variant'], 'ports': PORTS, 'model_id': MODEL_ID,
              'servers': [{'port': s['port'], 'gpu': s['gpu'], 'cmd': s['cmd'][1:], 'log': s['log']} for s in SPECS],
              'template_ok': TEMPLATE_OK, 'template_tails': {str(k): v[-300:] for k, v in TEMPLATE_TAILS.items()}}
print(f"server up SERVE_VARIANT={M['variant']} layout={M['layout']} ports={PORTS} model_id={MODEL_ID}", flush=True)
print(subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)


In [ ]:
# ===== 7. Run the grid (skip-if-done resume, deadline guard, heartbeat, traces) =====
from agentdojo.task_suite.load_suites import get_suites
from runners.main_grid import build_tasks, utility_task_list, run_grid, seed_from_inputs
from runners.run_benchmark import PINNED
OUT_JSONL = f'{OUTDIR}/{RUN_ID}.jsonl'
TRACE_DIR = f'{OUTDIR}/traces'
os.makedirs(OUTDIR, exist_ok=True)
SEEDED = seed_from_inputs(INPUT_ROOT, OUT_JSONL, TRACE_DIR, RUN_ID) if not os.path.exists(OUT_JSONL) else {}
print('resume seed from inputs:', {k: (v if not isinstance(v, list) else len(v)) for k, v in SEEDED.items()}, flush=True)
_suites = get_suites(BENCHMARK_VERSION)
UTIL_TASKS = utility_task_list(_suites) if RUN_UTILITY else []
TASKS = build_tasks(PAIRS, CONDITIONS, UTIL_TASKS, SHARD, N_SHARDS)
if SMOKE_N:   # smoke test only (never for the real run)
    TASKS = ([t for t in TASKS if t['kind'] == 'attack'][:SMOKE_N]
             + [t for t in TASKS if t['kind'] == 'utility'][:SMOKE_N])
print(f'{RUN_ID}: {len(TASKS)} episodes in this shard ({len(UTIL_TASKS)} utility in the full grid)', flush=True)
HEADER = {'run_id': RUN_ID, 'model_key': MODEL_KEY, 'subset': SUBSET, 'conditions': CONDITIONS,
          'run_utility': RUN_UTILITY, 'serve': SERVE_USED, 'gguf_sha256': GGUF_SHA, 'llama_build': LLAMA_BUILD,
          'seed': SEED, 'pinned': PINNED, 'embed_sha256': EMBED_SHA256, 'payload_sums_sha256': PAYLOAD_SUMS_SHA256}
STATS = run_grid(TASKS, PORTS, label=LABEL, out_jsonl=OUT_JSONL, trace_dir=TRACE_DIR, defense=DEFENSE,
                 streams_per_port=STREAMS_PER_SERVER, deadline_ts=SESSION_T0 + DEADLINE_H * 3600,
                 header=HEADER, shard=f'{SHARD}/{N_SHARDS}')
print('STATS', json.dumps({k: v for k, v in STATS.items() if k != 'worker_errors'}), flush=True)
if STATS['worker_errors']:
    print('WORKER ERRORS:', STATS['worker_errors'], flush=True)


In [ ]:
# ===== 8. Stop servers, manifest, zipped traces, MAIN_RESULT line =====
import datetime as _dt
from runners.main_grid import summarize, zip_traces, main_result_line
for _p in PROCS:
    try:
        _p.terminate()
    except Exception:
        pass
SUMMARY = summarize(OUT_JSONL, TASKS)
ZIP = zip_traces(TRACE_DIR, f'{WORK}/{RUN_ID}_traces.zip')
WALL_S = time.time() - SESSION_T0
MANIFEST = {
    'run_id': RUN_ID, 'model_key': MODEL_KEY, 'label': LABEL, 'shard': f'{SHARD}/{N_SHARDS}', 'subset': SUBSET,
    'n_pairs': len(PAIRS), 'conditions': CONDITIONS, 'defense': DEFENSE, 'run_utility': RUN_UTILITY,
    'streams_per_server': STREAMS_PER_SERVER, 'smoke_n': SMOKE_N,
    'gguf': {k: M[k] for k in ('repo', 'rev', 'file', 'size')} | {'sha256': GGUF_SHA},
    'serve': SERVE_USED, 'llama_build': LLAMA_BUILD, 'seed': SEED, 'pinned': PINNED,
    'embed': {'sha256': EMBED_SHA256, 'bytes': EMBED_BYTES, 'n_files': EMBED_N_FILES},
    'payload_verify': PAYLOAD_VERIFY, 'resume_seeded': SEEDED,
    'stats': STATS, 'summary': SUMMARY, 'traces_zip': ZIP, 'est_session_h': EST_SESSION_H,
    'session_started': _dt.datetime.fromtimestamp(SESSION_T0).isoformat(timespec='seconds'),
    'session_ended': _dt.datetime.now().isoformat(timespec='seconds'), 'wall_s': round(WALL_S, 1),
    'outputs': {'jsonl': OUT_JSONL, 'traces_dir': TRACE_DIR, 'traces_zip': ZIP['zip'],
                'server_logs': [s['log'] for s in SERVE_USED['servers']]},
}
MANIFEST_PATH = f'{WORK}/runs_manifest_{RUN_ID}.json'
Path(MANIFEST_PATH).write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print(json.dumps({k: MANIFEST[k] for k in ('run_id', 'summary', 'traces_zip', 'embed', 'wall_s')}, indent=1, default=str))
print(f'wrote {MANIFEST_PATH}, {OUT_JSONL}, {ZIP["zip"]} ({ZIP["n_traces"]} traces, {ZIP["zip_bytes"]} B)', flush=True)
if not SUMMARY['complete']:
    print(f"INCOMPLETE: {SUMMARY['missing']} missing + {SUMMARY['errors']} errored episodes. To resume, attach this "
          f"version's Output as an input (Add Input -> Notebook Output) and Save & Run All again (skip-if-done).", flush=True)
print(main_result_line(RUN_ID, LABEL, f'{SHARD}/{N_SHARDS}', SUMMARY, WALL_S, STATS, EMBED_SHA256), flush=True)


## After the run (on the box)
Download the Output (or `<run_id>_traces.zip` + `runs/<run_id>.jsonl` + `runs_manifest_<run_id>.json`), check the
`MAIN_RESULT` line and that `embed_sha256` matches the notebook header, then analyse with
`python -m analysis.run_analysis` (see `analysis/README.md`). Judge pass: `kaggle/JUDGE_ON_KAGGLE.md`.
Record slug/version, wall time, `MAIN_RESULT`, model id and GGUF sha256 in `DECISIONS.md`.